In [1]:
from model import EmotionCNN
from xAI.gradcam import  overlay_heatmap, gradcam
from xAI.smoothGrad import coumpute_smoothGrad
from xAI.occlusion import occlusion_saliency
from xAI.LayerActivation import get_conv_layer, get_layer_activation, layer_activation_heatmap_from_tensor
import torch
import torch.nn as nn
import cv2
import numpy as np
from torchvision import transforms
import matplotlib.pyplot as plt
from PIL import Image




/home/dorcas/miniconda3/envs/demo/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/home/dorcas/miniconda3/envs/demo/lib/python3.11/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/home/dorcas/miniconda3/envs/demo/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
idx_to_emotion = {
    0: "surprise",    
    1: "fear",        
    2: "disgust",    
    3: "happiness",   
    4: "sadness",     
    5: "anger",       
}


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EmotionCNN(num_classes=6).to(device)
WEIGHTS_PATH = "best_model_cosine.pt"
state = torch.load(WEIGHTS_PATH, map_location=device)
model.load_state_dict(state)
model.eval()





/tmp/ipykernel_10824/2803996623.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(WEIGHTS_PATH, map_location=device)


EmotionCNN(
  (features): Sequential(
    (0): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
      (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (7): Dropout2d(p=0.15, inplace=False)
    )
    (1): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, t

In [ ]:
# load image
img = Image.open("evaluation/angry.jpg")



transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])


tensor = transform(img).unsqueeze(0).to(device)



logits = model(tensor)
probs = torch.softmax(logits, dim=1)
conf, pred = torch.max(probs, dim=1)
pred_idx = int(pred.item())
conf= float(conf.item())
emotion = idx_to_emotion.get(pred_idx, str(pred_idx))
print("Emotion:", emotion, "Confidence:", conf)



img_cv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)


heatmap1 = gradcam(model, img_cv, pred_idx)
superimposed_img1 = overlay_heatmap(img_cv, heatmap1)
superimposed_img1 = cv2.cvtColor(superimposed_img1, cv2.COLOR_BGR2RGB)


heatmap2 = coumpute_smoothGrad(model, img_cv, pred_idx, 30)
superimposed_img2 = overlay_heatmap(img_cv, heatmap2)
superimposed_img2 = cv2.cvtColor(superimposed_img2, cv2.COLOR_BGR2RGB)

heatmap3 = occlusion_saliency(model, tensor, pred_idx)
superimposed_img3 = overlay_heatmap(img_cv, heatmap3.numpy())
superimposed_img3 = cv2.cvtColor(superimposed_img3, cv2.COLOR_BGR2RGB)


layer1 = get_conv_layer(model, which="last")
activation1 = get_layer_activation(model, layer1, tensor)
heatmap4 = layer_activation_heatmap_from_tensor(activation1)
heatmap4 = heatmap4.numpy()
superimposed_img4 = overlay_heatmap(img_cv, heatmap4)
superimposed_img4= cv2.cvtColor(superimposed_img4, cv2.COLOR_BGR2RGB)



plt.figure(figsize=(10,5))

plt.subplot(1,5,1)
plt.title("Original")
plt.imshow(img)
plt.text(
    0.5, -0.20,
    f"prediction: {emotion}\nConfidence: {conf:.2f}",
    ha="center",
    transform=plt.gca().transAxes
)
plt.axis('off')



plt.subplot(1,5,2)
plt.title("Grad-CAM")
plt.imshow(superimposed_img1)
plt.axis('off')

plt.subplot(1,5,3)
plt.title("SmoothGrad")
plt.imshow(superimposed_img2)
plt.axis('off')

plt.subplot(1,5,4)
plt.title("Occlusion")
plt.imshow(superimposed_img3)
plt.axis('off')

plt.subplot(1,5,5)
plt.title("LayerActivation")
plt.imshow(superimposed_img4)
plt.axis('off')

plt.show()








Emotion: anger Confidence: 0.6871216893196106
